# 01 — Data Exploration

Initial exploration of the CRMLS sold-listings data (Jan 2022 – present).

Raw CSVs live in `Data/`, which is git-ignored and never committed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

DATA_DIR = Path("Data")

## Load data

Each file is one month of closed listings (the first file covers 2022–2023). Some months have a `_filled` variant; we load every CSV and tag rows with their source file.

In [ ]:
files = sorted(DATA_DIR.glob("CRMLSSold*.csv"))
print(f"{len(files)} files")

frames = []
for f in files:
    part = pd.read_csv(f, low_memory=False)
    part["source_file"] = f.name
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
print(df.shape)

In [ ]:
df.groupby("source_file").size()

## Structure

In [ ]:
df.info(memory_usage="deep")

In [ ]:
df.head()

### Duplicates

In [ ]:
print("Duplicate ListingKey rows:", df["ListingKey"].duplicated().sum())

## Missing values

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].to_frame("pct_missing").style.format("{:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 14))
missing.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel("Fraction missing")
ax.set_title("Missing values by column")
plt.tight_layout()

## Categorical overview

In [ ]:
for col in ["PropertyType", "PropertySubType", "MlsStatus", "CountyOrParish"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))

## Numeric summary

In [ ]:
num_cols = ["ClosePrice", "ListPrice", "OriginalListPrice", "LivingArea", "LotSizeSquareFeet",
            "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt", "DaysOnMarket", "AssociationFee"]
df[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T

## Focus: residential sales

`ResidentialLease` rows are rentals (ClosePrice is monthly rent), so price analysis uses `Residential` only.

In [ ]:
res = df[df["PropertyType"] == "Residential"].copy()
res["CloseDate"] = pd.to_datetime(res["CloseDate"], errors="coerce")
res["price_per_sqft"] = res["ClosePrice"] / res["LivingArea"].where(res["LivingArea"] > 0)
res["sale_to_list"] = res["ClosePrice"] / res["ListPrice"].where(res["ListPrice"] > 0)
print(res.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
clipped = res["ClosePrice"].clip(upper=res["ClosePrice"].quantile(0.99))
sns.histplot(clipped, bins=80, ax=axes[0])
axes[0].set_title("ClosePrice (clipped at 99th pct)")
sns.histplot(np.log10(res.loc[res["ClosePrice"] > 0, "ClosePrice"]), bins=80, ax=axes[1])
axes[1].set_title("log10(ClosePrice)")
plt.tight_layout()

### Trends over time

In [ ]:
monthly = (res.set_index("CloseDate")
             .resample("MS")
             .agg(sales=("ClosePrice", "size"),
                  median_price=("ClosePrice", "median"),
                  median_dom=("DaysOnMarket", "median")))

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
monthly["sales"].plot(ax=axes[0], title="Closed sales per month")
monthly["median_price"].plot(ax=axes[1], title="Median close price")
monthly["median_dom"].plot(ax=axes[2], title="Median days on market")
plt.tight_layout()

### Price by county

In [ ]:
top_counties = res["CountyOrParish"].value_counts().head(10).index
(res[res["CountyOrParish"].isin(top_counties)]
    .groupby("CountyOrParish")[["ClosePrice", "price_per_sqft", "DaysOnMarket"]]
    .median()
    .sort_values("ClosePrice", ascending=False))

### Correlations

In [ ]:
corr_cols = ["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
             "YearBuilt", "LotSizeSquareFeet", "DaysOnMarket", "sale_to_list"]
plt.figure(figsize=(8, 6))
sns.heatmap(res[corr_cols].corr(method="spearman"), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Spearman correlation (Residential)")
plt.tight_layout()

## Notes / next steps

- Check whether `_filled` files overlap with other months (duplicate `ListingKey`s).
- Decide on outlier rules for price, living area, and lot size.
- Choose which high-missingness columns to drop or impute.